# Tópicos Prácticos en Minería de Datos

**Universidad de los Andes — Minería de Datos**  
**Semana 12 · Sesión 23 — Tópicos Avanzados**

---

Este cuaderno cierra el curso integrando los conceptos de las 12 semanas y presenta tópicos avanzados de uso profesional: pipelines completos, selección estadística de modelos, interpretabilidad, detección de data drift y buenas prácticas. Culmina con un mini-proyecto integrador que recorre el ciclo completo de un proyecto de minería de datos.

## 1. Introducción y Objetivos

### ¿Qué cubriremos?

| # | Tema | Conexión con el curso |
|---|------|-----------------------|
| 1 | Pipelines end-to-end | Semanas 1, 3, 7 |
| 2 | Selección y comparación de modelos | Semanas 2–6 |
| 3 | Interpretabilidad | Semanas 5, 6 |
| 4 | Data drift | Semanas 1, 3 |
| 5 | Buenas prácticas y errores comunes | Todo el curso |
| 6 | Mini-proyecto integrador | Todo el curso |

### Objetivos de aprendizaje

Al finalizar este cuaderno el estudiante podrá:

1. Construir y serializar un pipeline completo listo para producción.
2. Comparar modelos con rigor estadístico.
3. Explicar predicciones con herramientas model-agnostic.
4. Identificar data leakage y otros errores frecuentes antes de que ocurran.
5. Articular todos los pasos del ciclo de vida de un modelo en un proyecto cohesivo.

## Importaciones globales

Todas las librerías necesarias se importan aquí para mantener el cuaderno autocontenido.

In [ ]:
# ── Utilidades generales ──────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import joblib
import os
import tempfile
from scipy import stats

# ── Visualización ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Datos ─────────────────────────────────────────────────────────────────────
from sklearn.datasets import (
    fetch_california_housing,
    load_breast_cancer,
    load_wine,
    make_classification,
    make_regression,
)

# ── Preprocesamiento ──────────────────────────────────────────────────────────
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, LabelEncoder,
    OrdinalEncoder, TargetEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ── Modelos ───────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    RandomForestRegressor
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# ── Evaluación ────────────────────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    StratifiedKFold, learning_curve, validation_curve
)
from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_auc_score, mean_squared_error, r2_score
)

# ── Interpretabilidad ─────────────────────────────────────────────────────────
from sklearn.inspection import (
    permutation_importance,
    PartialDependenceDisplay
)

# ── Configuración global ──────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid', palette='muted')

print('Todas las librerías cargadas correctamente ✓')

---

## 2. Pipelines Completos End-to-End

Un **pipeline** encadena transformaciones y un estimador en un solo objeto que:

- Evita data leakage al aplicar transformaciones solo con estadísticas del set de entrenamiento.
- Simplifica la validación cruzada y la búsqueda de hiperparámetros.
- Facilita el despliegue: se serializa y deserializa en un solo archivo.

```
Datos crudos → [Imputación] → [Escalado / Encoding] → [Modelo] → Predicciones
```

### 2.1 Preparación del dataset

Usaremos **California Housing** (regresión) e introduciremos columnas categóricas sintéticas para mostrar `ColumnTransformer`.

In [ ]:
# ── Carga del dataset ─────────────────────────────────────────────────────────
housing = fetch_california_housing(as_frame=True)
df_raw = housing.frame.copy()

# ── Crear variables categóricas sintéticas para el ejemplo ────────────────────
rng = np.random.default_rng(42)
df_raw['ocean_proximity'] = rng.choice(
    ['INLAND', 'NEAR BAY', '<1H OCEAN', 'NEAR OCEAN'], size=len(df_raw)
)

# ── Introducir valores faltantes sintéticos (~5%) ─────────────────────────────
for col in ['MedInc', 'HouseAge', 'AveRooms']:
    mask = rng.random(len(df_raw)) < 0.05
    df_raw.loc[mask, col] = np.nan

target_col = 'MedHouseVal'
feature_cols = [c for c in df_raw.columns if c != target_col]

print(f'Shape: {df_raw.shape}')
print(f'Nulos por columna:\n{df_raw.isnull().sum()[df_raw.isnull().sum() > 0]}')
df_raw[feature_cols].head(3)

### 2.2 Construcción del pipeline con `ColumnTransformer`

In [ ]:
# ── Separar columnas numéricas y categóricas ──────────────────────────────────
num_cols = df_raw[feature_cols].select_dtypes(include='number').columns.tolist()
cat_cols = df_raw[feature_cols].select_dtypes(include='object').columns.tolist()

print(f'Numéricas : {num_cols}')
print(f'Categóricas: {cat_cols}')

# ── Sub-pipeline numérico: imputación mediana + escalado estándar ─────────────
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # maneja NaN
    ('scaler',  StandardScaler()),                   # media 0, std 1
])

# ── Sub-pipeline categórico: imputación moda + One-Hot Encoding ───────────────
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value',
                               unknown_value=-1)),
])

# ── ColumnTransformer: combina ambos sub-pipelines ────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,    num_cols),
    ('cat', categorical_transformer, cat_cols),
], remainder='drop')   # descarta columnas no especificadas

# ── Pipeline completo: preprocesamiento + modelo ──────────────────────────────
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
])

print('\nEstructura del pipeline:')
print(pipeline_rf)

### 2.3 Entrenamiento, evaluación y serialización

In [ ]:
# ── Split train/test ──────────────────────────────────────────────────────────
X = df_raw[feature_cols]
y = df_raw[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── Entrenamiento ─────────────────────────────────────────────────────────────
pipeline_rf.fit(X_train, y_train)

# ── Evaluación en test ────────────────────────────────────────────────────────
y_pred = pipeline_rf.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'RMSE en test : {rmse:.4f}')
print(f'R²   en test : {r2:.4f}')

# ── Serialización con joblib ──────────────────────────────────────────────────
model_path = os.path.join(tempfile.gettempdir(), 'pipeline_rf_housing.pkl')
joblib.dump(pipeline_rf, model_path)         # guardar
pipeline_loaded = joblib.load(model_path)    # cargar

# Verificar que el modelo cargado predice igual
y_pred_loaded = pipeline_loaded.predict(X_test)
assert np.allclose(y_pred, y_pred_loaded), 'Las predicciones difieren!'
print(f'\nModelo serializado en: {model_path}')
print('Predicciones idénticas tras serialización ✓')

### 2.4 Validación cruzada dentro del pipeline

> **Clave**: `cross_val_score` aplica el pipeline completo en cada fold, de modo que el escalado y la imputación nunca ven datos de test.

In [ ]:
# ── CV de 5 folds sobre el pipeline completo ─────────────────────────────────
cv_scores = cross_val_score(
    pipeline_rf, X_train, y_train,
    cv=5, scoring='r2', n_jobs=-1
)

print('R² por fold:', np.round(cv_scores, 4))
print(f'Media CV R² : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# ── Visualización ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(range(1, 6), cv_scores, color='steelblue', alpha=0.8, edgecolor='navy')
ax.axhline(cv_scores.mean(), color='crimson', ls='--', lw=1.5,
           label=f'Media = {cv_scores.mean():.3f}')
ax.set(xlabel='Fold', ylabel='R²', title='Validación cruzada — Random Forest (Housing)',
       ylim=(0.5, 1.0), xticks=range(1, 6))
ax.legend()
plt.tight_layout()
plt.show()

---

## 3. Selección y Comparación de Modelos

Elegir el modelo "correcto" requiere más que una sola métrica en test. Se deben considerar:

- **Rendimiento promedio** y **varianza** en validación cruzada.
- **Significancia estadística** de las diferencias (t-test pareado de Dietterich, 1998).
- **Curvas de aprendizaje** (bias vs. variance).
- **Costo computacional** y **interpretabilidad** según el contexto.

### 3.1 Benchmark sistemático de clasificadores

Usaremos **Breast Cancer Wisconsin** (clasificación binaria).

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
bc = load_breast_cancer(as_frame=True)
X_bc, y_bc = bc.data, bc.target

# ── Modelos a comparar (cada uno envuelto en pipeline con escalado) ────────────
classifiers = {
    'Regresión Logística': Pipeline([('sc', StandardScaler()),
                                     ('clf', LogisticRegression(max_iter=1000))]),
    'KNN (k=5)':           Pipeline([('sc', StandardScaler()),
                                     ('clf', KNeighborsClassifier(n_neighbors=5))]),
    'Árbol de Decisión':   Pipeline([('sc', StandardScaler()),
                                     ('clf', DecisionTreeClassifier(max_depth=6, random_state=42))]),
    'Random Forest':       Pipeline([('sc', StandardScaler()),
                                     ('clf', RandomForestClassifier(n_estimators=100, random_state=42))]),
    'Gradient Boosting':   Pipeline([('sc', StandardScaler()),
                                     ('clf', GradientBoostingClassifier(n_estimators=100, random_state=42))]),
    'SVM (RBF)':           Pipeline([('sc', StandardScaler()),
                                     ('clf', SVC(kernel='rbf', probability=True, random_state=42))]),
    'Naive Bayes':         Pipeline([('sc', StandardScaler()),
                                     ('clf', GaussianNB())]),
}

# ── Validación cruzada estratificada (10 folds) ───────────────────────────────
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
results = {}

for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_bc, y_bc, cv=skf,
                             scoring='roc_auc', n_jobs=-1)
    results[name] = scores
    print(f'{name:<25}  AUC = {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# ── Tabla resumen de métricas ─────────────────────────────────────────────────
df_results = pd.DataFrame(results).T
df_results.columns = [f'Fold {i+1}' for i in range(10)]
df_results['Media'] = df_results.mean(axis=1)
df_results['Std']   = df_results.std(axis=1)
df_results['IC 95% inf'] = df_results['Media'] - 1.96 * df_results['Std']
df_results['IC 95% sup'] = df_results['Media'] + 1.96 * df_results['Std']

display(df_results[['Media', 'Std', 'IC 95% inf', 'IC 95% sup']].sort_values('Media', ascending=False).round(4))

In [ ]:
# ── Boxplot comparativo ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4.5))

order = df_results['Media'].sort_values(ascending=False).index.tolist()
data_plot = [results[name] for name in order]

bp = ax.boxplot(data_plot, labels=order, patch_artist=True, notch=True,
                medianprops=dict(color='black', linewidth=2))

colors = plt.cm.Blues(np.linspace(0.4, 0.85, len(order)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set(ylabel='AUC-ROC (10-fold CV)',
       title='Comparación de clasificadores — Breast Cancer',
       ylim=(0.88, 1.02))
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### 3.2 Significancia estadística: t-test pareado

Dado que los pliegues son los mismos para todos los modelos, podemos aplicar un **t-test pareado** para determinar si las diferencias de rendimiento son estadísticamente significativas.

> **Hipótesis**  
> H₀: los dos modelos tienen el mismo AUC esperado  
> H₁: los AUC esperados son distintos  
> Rechazamos H₀ si p-value < 0.05

In [ ]:
# ── t-test pareado entre el mejor y el resto ──────────────────────────────────
best_name = df_results['Media'].idxmax()
best_scores = results[best_name]

print(f'Modelo de referencia: {best_name} (AUC media = {best_scores.mean():.4f})')
print('─' * 65)
print(f'{"Comparado con":<25}  {"t-stat":>8}  {"p-value":>10}  {"Significativo":>14}')
print('─' * 65)

for name, scores in results.items():
    if name == best_name:
        continue
    t_stat, p_val = stats.ttest_rel(best_scores, scores)
    sig = '✓ (p<0.05)' if p_val < 0.05 else '✗'
    print(f'{name:<25}  {t_stat:>8.3f}  {p_val:>10.4f}  {sig:>14}')

### 3.3 Curvas de aprendizaje — Bias vs. Varianza

Las **curvas de aprendizaje** muestran cómo evoluciona el error de entrenamiento y validación a medida que aumenta el tamaño del conjunto de entrenamiento.

| Patrón | Diagnóstico | Solución |
|--------|-------------|----------|
| Error train alto, error val alto | Alto bias (underfitting) | Modelo más complejo, más features |
| Error train bajo, error val alto | Alta varianza (overfitting) | Más datos, regularización, simplificar |

In [ ]:
def plot_learning_curve(estimator, title, X, y, ax, cv=5, n_jobs=-1):
    """Traza la curva de aprendizaje sobre un eje dado."""
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=n_jobs,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='roc_auc'
    )
    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    test_mean  = test_scores.mean(axis=1)
    test_std   = test_scores.std(axis=1)

    ax.fill_between(train_sizes, train_mean - train_std,
                    train_mean + train_std, alpha=0.15, color='royalblue')
    ax.fill_between(train_sizes, test_mean - test_std,
                    test_mean + test_std,  alpha=0.15, color='darkorange')
    ax.plot(train_sizes, train_mean, 'o-', color='royalblue',  label='Entrenamiento')
    ax.plot(train_sizes, test_mean,  'o-', color='darkorange', label='Validación CV')
    ax.set(title=title, xlabel='Muestras de entrenamiento', ylabel='AUC-ROC')
    ax.legend(fontsize=9)
    ax.set_ylim(0.7, 1.02)


# ── Dos modelos representativos ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

plot_learning_curve(
    Pipeline([('sc', StandardScaler()),
              ('clf', DecisionTreeClassifier(max_depth=20, random_state=42))]),
    'Árbol profundo (alta varianza)',
    X_bc, y_bc, axes[0]
)

plot_learning_curve(
    Pipeline([('sc', StandardScaler()),
              ('clf', LogisticRegression(C=0.01, max_iter=1000))]),
    'Regresión Logística muy regularizada (alto bias)',
    X_bc, y_bc, axes[1]
)

plt.suptitle('Curvas de aprendizaje — Breast Cancer', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 3.4 Curva de validación — tuning de un hiperparámetro

In [ ]:
# ── Curva de validación para max_depth en Random Forest ──────────────────────
param_range = [2, 4, 6, 8, 12, 16, 20, None]

train_scores_vc, test_scores_vc = validation_curve(
    RandomForestClassifier(n_estimators=50, random_state=42),
    X_bc, y_bc,
    param_name='max_depth',
    param_range=param_range,
    cv=5, scoring='roc_auc', n_jobs=-1
)

fig, ax = plt.subplots(figsize=(8, 4))
x_labels = [str(d) if d else 'None' for d in param_range]
x_pos = range(len(param_range))

ax.fill_between(x_pos,
                train_scores_vc.mean(1) - train_scores_vc.std(1),
                train_scores_vc.mean(1) + train_scores_vc.std(1), alpha=0.15, color='royalblue')
ax.fill_between(x_pos,
                test_scores_vc.mean(1)  - test_scores_vc.std(1),
                test_scores_vc.mean(1)  + test_scores_vc.std(1),  alpha=0.15, color='darkorange')
ax.plot(x_pos, train_scores_vc.mean(1), 'o-', color='royalblue',  label='Entrenamiento')
ax.plot(x_pos, test_scores_vc.mean(1),  'o-', color='darkorange', label='Validación CV')

best_idx = test_scores_vc.mean(1).argmax()
ax.axvline(best_idx, color='green', ls='--', alpha=0.8,
           label=f'Mejor: max_depth={x_labels[best_idx]}')

ax.set(xticks=x_pos, xticklabels=x_labels,
       xlabel='max_depth', ylabel='AUC-ROC',
       title='Curva de validación — Random Forest (max_depth)')
ax.legend()
plt.tight_layout()
plt.show()

---

## 4. Interpretabilidad de Modelos

Un modelo con buenas métricas pero que nadie entiende es difícil de mantener, auditar y defender ante usuarios. La interpretabilidad va desde métodos **intrínsecos** (árboles, coeficientes) hasta métodos **post-hoc model-agnostic** (permutation importance, PDP, SHAP).

```
Nivel de interpretabilidad
├── Intrínseca      → coeficientes (lineal), feature_importances_ (árbol)
└── Post-hoc
    ├── Global      → Permutation Importance, PDP, SHAP summary
    └── Local       → LIME, SHAP waterfall / force plot
```

### 4.1 Feature importance intrínseca (árboles)

In [ ]:
# ── Entrenamos un Random Forest sobre Breast Cancer ───────────────────────────
X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    X_bc, y_bc, test_size=0.25, random_state=42, stratify=y_bc
)

rf_bc = RandomForestClassifier(n_estimators=200, random_state=42)
rf_bc.fit(X_train_bc, y_train_bc)

# ── Importancia por reducción de impureza (MDI) ───────────────────────────────
importances_mdi = pd.Series(
    rf_bc.feature_importances_,
    index=bc.feature_names
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
importances_mdi.head(15).sort_values().plot.barh(ax=ax, color='steelblue', edgecolor='navy')
ax.set(title='Feature Importance (MDI) — Top 15 variables',
       xlabel='Importancia media por reducción de impureza')
plt.tight_layout()
plt.show()

print('⚠  Advertencia: MDI puede sobreestimar variables continuas con muchos valores únicos.')

### 4.2 Permutation Importance (model-agnostic)

La **importancia por permutación** mide cuánto baja el rendimiento del modelo cuando se aleatoriza una variable. Es válida para **cualquier** modelo y se evalúa en el **conjunto de test** (no de entrenamiento).

In [ ]:
# ── Permutation importance en datos de test ───────────────────────────────────
perm_imp = permutation_importance(
    rf_bc, X_test_bc, y_test_bc,
    n_repeats=20,          # número de permutaciones por feature
    random_state=42,
    scoring='roc_auc',
    n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature':   bc.feature_names,
    'importance': perm_imp.importances_mean,
    'std':        perm_imp.importances_std
}).sort_values('importance', ascending=False)

# ── Comparación MDI vs. Permutation Importance ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MDI
importances_mdi.head(15).sort_values().plot.barh(
    ax=axes[0], color='steelblue', edgecolor='navy'
)
axes[0].set(title='Importancia MDI (entrenamiento)',
            xlabel='Importancia')

# Permutation
top15_perm = perm_df.head(15).sort_values('importance')
axes[1].barh(top15_perm['feature'], top15_perm['importance'],
             xerr=top15_perm['std'], color='darkorange',
             edgecolor='saddlebrown', capsize=4)
axes[1].set(title='Permutation Importance (test)',
            xlabel='Reducción en AUC-ROC')
axes[1].axvline(0, color='black', lw=0.8, ls='--')

plt.suptitle('MDI vs. Permutation Importance — Random Forest (Breast Cancer)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### 4.3 Partial Dependence Plots (PDP)

Los **PDP** muestran el efecto marginal promedio de una variable sobre la predicción del modelo, manteniendo todas las demás variables en sus valores observados. Son útiles para detectar relaciones no lineales y efectos de interacción.

In [ ]:
# ── Variables con mayor permutation importance ────────────────────────────────
top2_features = perm_df.head(2)['feature'].tolist()
top2_idx = [list(bc.feature_names).index(f) for f in top2_features]

print(f'Variables seleccionadas para PDP: {top2_features}')

# ── PDP individual + PDP de interacción ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# PDP feature 1
PartialDependenceDisplay.from_estimator(
    rf_bc, X_test_bc, features=[top2_idx[0]],
    feature_names=bc.feature_names,
    ax=axes[0], grid_resolution=50
)
axes[0].set_title(f'PDP — {top2_features[0]}')

# PDP feature 2
PartialDependenceDisplay.from_estimator(
    rf_bc, X_test_bc, features=[top2_idx[1]],
    feature_names=bc.feature_names,
    ax=axes[1], grid_resolution=50
)
axes[1].set_title(f'PDP — {top2_features[1]}')

# PDP de interacción (2D)
PartialDependenceDisplay.from_estimator(
    rf_bc, X_test_bc, features=[(top2_idx[0], top2_idx[1])],
    feature_names=bc.feature_names,
    ax=axes[2], grid_resolution=20
)
axes[2].set_title(f'PDP interacción')

plt.suptitle('Partial Dependence Plots — Random Forest (Breast Cancer)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### 4.4 Aproximación manual a SHAP values

**SHAP** (SHapley Additive exPlanations) asigna a cada feature una contribución al valor de predicción de una instancia particular, basada en la teoría de juegos cooperativos.

La librería `shap` (cuando está disponible) calcula estos valores eficientemente. Aquí mostramos la **interpretación conceptual** usando el TreeExplainer de sklearn cuando `shap` no está instalado, y usamos la librería completa si está disponible.

In [ ]:
# ── Intentar usar shap; si no está disponible, construimos una aproximación ───
try:
    import shap
    SHAP_AVAILABLE = True
    print('Librería shap disponible ✓')
except ImportError:
    SHAP_AVAILABLE = False
    print('shap no instalado — usando aproximación basada en Permutation Importance')

if SHAP_AVAILABLE:
    # ── TreeExplainer es exacto y rápido para modelos basados en árboles ─────
    explainer   = shap.TreeExplainer(rf_bc)
    shap_values = explainer.shap_values(X_test_bc)

    # Para clasificación binaria shap_values es una lista [clase0, clase1]
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values

    # ── Beeswarm (summary plot) ───────────────────────────────────────────────
    plt.figure(figsize=(9, 6))
    shap.summary_plot(sv, X_test_bc,
                      feature_names=list(bc.feature_names),
                      show=False)
    plt.title('SHAP Summary Plot — Random Forest (Breast Cancer)')
    plt.tight_layout()
    plt.show()

    # ── Explicación local: instancia 0 ────────────────────────────────────────
    print('\nExplicación local para la primera instancia de test:')
    shap_exp = shap.Explanation(
        values=sv[0],
        base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                    else explainer.expected_value,
        data=X_test_bc.iloc[0].values,
        feature_names=list(bc.feature_names)
    )
    shap.plots.waterfall(shap_exp, show=False)
    plt.tight_layout()
    plt.show()

else:
    # ── Aproximación: SHAP global ≈ permutation importance con signo ──────────
    print('\nAproximación: ranking de variables por impacto en la predicción')
    display(perm_df.head(10).reset_index(drop=True))

**Lectura del SHAP summary plot:**

- Cada punto representa una instancia en el conjunto de evaluación.
- El **eje X** indica la contribución de esa feature a la predicción (positivo → más probable la clase 1).
- El **color** representa el valor original de la feature (rojo = alto, azul = bajo).
- Las features se ordenan de mayor a menor impacto global.

---

## 5. Detección y Manejo de Data Drift

Un modelo entrenado hoy puede degradarse con el tiempo porque los datos de producción cambian. Distinguimos dos tipos:

| Tipo | Definición | Detección |
|------|-----------|----------|
| **Data drift** (covariable) | P(X) cambia, P(Y\|X) estable | Tests estadísticos sobre X |
| **Concept drift** | P(Y\|X) cambia | Monitoreo de métricas del modelo |
| **Label drift** | P(Y) cambia | Comparar distribución de Y |

In [ ]:
# ── Simular data drift ────────────────────────────────────────────────────────
rng2 = np.random.default_rng(99)

# Distribución original (entrenamiento)
feature_ref = rng2.normal(loc=0.0, scale=1.0, size=500)   # media 0
# Distribución con drift (producción) — media desplazada
feature_new = rng2.normal(loc=0.8, scale=1.2, size=500)   # media 0.8, mayor dispersión

# ── Tests estadísticos ────────────────────────────────────────────────────────
# 1) Kolmogorov-Smirnov: compara distribuciones acumuladas
ks_stat, ks_pval = stats.ks_2samp(feature_ref, feature_new)

# 2) Mann-Whitney U: no paramétrico, compara medianas
mw_stat, mw_pval = stats.mannwhitneyu(feature_ref, feature_new, alternative='two-sided')

print('Test KS       → estadístico={:.4f}, p-value={:.4e}'.format(ks_stat, ks_pval))
print('Mann-Whitney  → estadístico={:.1f}, p-value={:.4e}'.format(mw_stat, mw_pval))

if ks_pval < 0.05:
    print('\n⚠  DATA DRIFT DETECTADO: las distribuciones difieren significativamente.')
else:
    print('\n✓  Sin drift detectado.')

# ── Visualización de distribuciones ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogramas superpuestos
axes[0].hist(feature_ref, bins=30, alpha=0.6, label='Referencia (train)', color='steelblue')
axes[0].hist(feature_new, bins=30, alpha=0.6, label='Producción (nuevo)', color='darkorange')
axes[0].set(title='Distribución de una feature', xlabel='Valor', ylabel='Frecuencia')
axes[0].legend()

# CDF (para visualizar KS)
ref_sorted = np.sort(feature_ref)
new_sorted = np.sort(feature_new)
axes[1].plot(ref_sorted, np.arange(1, 501)/500, label='Referencia', color='steelblue')
axes[1].plot(new_sorted, np.arange(1, 501)/500, label='Producción',  color='darkorange')
axes[1].set(title='Función de distribución acumulada (CDF)',
            xlabel='Valor', ylabel='Probabilidad acumulada')
axes[1].legend()
axes[1].text(0.05, 0.8,
             f'KS stat = {ks_stat:.3f}\np-value = {ks_pval:.2e}',
             transform=axes[1].transAxes, fontsize=9,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Detección de Data Drift — Test KS', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Simulación de degradación del modelo a lo largo del tiempo ────────────────
# Entrenamos con datos sin drift y evaluamos en batches con drift creciente

X_train_drift, y_train_drift = make_classification(
    n_samples=1000, n_features=10, random_state=1
)
clf_drift = RandomForestClassifier(n_estimators=50, random_state=1)
clf_drift.fit(X_train_drift, y_train_drift)

drift_levels = np.linspace(0, 3.0, 10)   # nivel de desplazamiento de la media
auc_drift    = []

for level in drift_levels:
    rng_d = np.random.default_rng(int(level * 100))
    X_prod = X_train_drift.copy() + rng_d.normal(loc=level, scale=0.5,
                                                  size=X_train_drift.shape)
    # Etiquetas no cambian (solo data drift, no concept drift)
    _, X_batch, _, y_batch = train_test_split(X_prod, y_train_drift, test_size=0.3, random_state=42)
    auc = roc_auc_score(y_batch, clf_drift.predict_proba(X_batch)[:, 1])
    auc_drift.append(auc)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(drift_levels, auc_drift, 'o-', color='crimson', lw=2)
ax.axhline(0.5, color='gray', ls='--', label='Modelo aleatorio (AUC=0.5)')
ax.fill_between(drift_levels, 0.5, auc_drift, alpha=0.15, color='crimson')
ax.set(xlabel='Nivel de drift (desplazamiento de la media)',
       ylabel='AUC-ROC en producción',
       title='Degradación del modelo con data drift creciente')
ax.legend()
plt.tight_layout()
plt.show()

---

## 6. Buenas Prácticas y Errores Comunes

Esta sección resume los errores más frecuentes en proyectos de minería de datos reales.

### 6.1 Data Leakage — Ejemplo concreto

**Data leakage** ocurre cuando información del conjunto de test (o del futuro) contamina el entrenamiento, produciendo métricas engañosamente buenas.

In [ ]:
# ── Dataset sintético con 20 features de ruido puro ──────────────────────────
rng_leak = np.random.default_rng(0)
n, p = 100, 200    # pocas muestras, muchas features → escenario de alta dimensión
X_leak = rng_leak.standard_normal((n, p))
y_leak = rng_leak.integers(0, 2, size=n)

# ─────────────────────────────────────────────────────────────────────────────
# INCORRECTO: selección de features ANTES del split → leakage
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.feature_selection import SelectKBest, f_classif

# 1) Seleccionar features con TODOS los datos (incluido el futuro test set)
selector_leaky = SelectKBest(f_classif, k=10)
X_leak_selected = selector_leaky.fit_transform(X_leak, y_leak)   # ← usa y de test!

# 2) Dividir DESPUÉS de la selección
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_leak_selected, y_leak,
                                               test_size=0.3, random_state=0)
clf_leaky = LogisticRegression(max_iter=200)
clf_leaky.fit(Xl_tr, yl_tr)
acc_leaky = accuracy_score(yl_te, clf_leaky.predict(Xl_te))

# ─────────────────────────────────────────────────────────────────────────────
# CORRECTO: selección de features DENTRO del pipeline → sin leakage
# ─────────────────────────────────────────────────────────────────────────────
Xl2_tr, Xl2_te, yl2_tr, yl2_te = train_test_split(X_leak, y_leak,
                                                   test_size=0.3, random_state=0)
pipe_correct = Pipeline([
    ('selector', SelectKBest(f_classif, k=10)),   # fit solo en train
    ('clf',      LogisticRegression(max_iter=200))
])
pipe_correct.fit(Xl2_tr, yl2_tr)
acc_correct = accuracy_score(yl2_te, pipe_correct.predict(Xl2_te))

print(f'Exactitud con LEAKAGE   : {acc_leaky:.3f}  ← inflada artificialmente')
print(f'Exactitud SIN leakage   : {acc_correct:.3f}  ← estimación real')
print(f'Diferencia              : {acc_leaky - acc_correct:+.3f}')

### 6.2 Pitfalls del Target Encoding

El **target encoding** reemplaza categorías por la media del target en esa categoría. Si no se aplica con cross-fitting, produce leakage.

In [ ]:
# ── Ejemplo de target encoding con y sin protección ──────────────────────────
rng_te = np.random.default_rng(7)
n_te   = 500

df_te = pd.DataFrame({
    'categoria': rng_te.choice(['A', 'B', 'C', 'D'], size=n_te),
    'feature_num': rng_te.standard_normal(n_te),
    'target': rng_te.integers(0, 2, size=n_te)
})

# INCORRECTO: calcular la media del target con TODOS los datos
target_means_leaky = df_te.groupby('categoria')['target'].mean()
df_te['cat_encoded_leaky'] = df_te['categoria'].map(target_means_leaky)

# CORRECTO: sklearn TargetEncoder usa cross-fitting internamente
X_te = df_te[['categoria', 'feature_num']]
y_te = df_te['target']

X_te_train, X_te_test, y_te_train, y_te_test = train_test_split(
    X_te, y_te, test_size=0.3, random_state=7
)

pipe_te = Pipeline([
    ('enc', ColumnTransformer([
        ('te', TargetEncoder(smooth='auto'), ['categoria']),
        ('num', 'passthrough', ['feature_num'])
    ])),
    ('clf', LogisticRegression())
])
pipe_te.fit(X_te_train, y_te_train)
print(f'AUC con TargetEncoder correcto: {roc_auc_score(y_te_test, pipe_te.predict_proba(X_te_test)[:, 1]):.4f}')

print('\nReglas para usar Target Encoding de forma segura:')
print('  1. Siempre dentro de un pipeline (fit solo en train).')
print('  2. Usar smoothing o regularización para categorías poco frecuentes.')
print('  3. Preferir sklearn TargetEncoder que implementa cross-fitting automático.')

### 6.3 Checklist anti-overfitting

```
□ La métrica de test está muy por encima de la métrica de CV → sospecha leakage
□ Error de entrenamiento ≈ 0, error de validación alto → overfitting clásico
□ Se usaron datos de test para elegir hiperparámetros → doble dipping
□ Hay columnas derivadas del target en las features
□ Los datos de tiempo no se dividieron cronológicamente
□ La imputación se ajustó sobre todo el dataset antes del split
□ El escalado se ajustó sobre todo el dataset antes del split
```

### 6.4 Guía de selección de algoritmo

| Situación | Algoritmos recomendados |
|-----------|------------------------|
| Baseline rápido y explicable | Regresión Logística / Ridge |
| Alta dimensionalidad (p >> n) | Lasso, Elastic Net, SVM lineal |
| Relaciones no lineales complejas | Random Forest, Gradient Boosting |
| Requiere calibración de probabilidades | Regresión Logística, calibrado post-hoc |
| Dataset pequeño (< 1 000) | KNN, SVM, métodos de ensemble con CV |
| Dataset enorme (> 1 M) | SGD, Gradient Boosting incremental |
| Interpretabilidad regulatoria | Regresión lineal, árbol pequeño |
| Datos mixtos (numérico + categórico) | Gradient Boosting (nativo), CatBoost |
| Anomaly detection | Isolation Forest, LOF, Autoencoder |
| Clustering | K-Means (convexo), DBSCAN (forma libre) |

---

## 7. Repaso Integrador — Mini-Proyecto Completo

Este mini-proyecto recorre el ciclo completo de un proyecto de minería de datos usando el dataset **Wine** (clasificación multiclase). Integra conceptos de las 12 semanas:

**Semana 1** Carga y preprocesamiento → **Semana 2–4** Modelos lineales y regularización → **Semana 5–6** SVM, Árboles, Ensambles → **Semana 3** Evaluación → **Semana 7** Feature Engineering → **Semanas 8–11** EDA y reducción → **Semana 12** Interpretabilidad y buenas prácticas

### 7.1 Carga del dataset y EDA

In [ ]:
# ── Dataset Wine ──────────────────────────────────────────────────────────────
wine = load_wine(as_frame=True)
df_wine = wine.frame.copy()
df_wine.rename(columns={'target': 'clase'}, inplace=True)

print('Shape:', df_wine.shape)
print('\nDistribución de clases:')
print(df_wine['clase'].value_counts().rename({0:'Clase 0', 1:'Clase 1', 2:'Clase 2'}))
df_wine.describe().round(2)

In [ ]:
# ── EDA: distribuciones y correlaciones ───────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig)

# Distribución de algunas features por clase
ax1 = fig.add_subplot(gs[0, 0])
for cls in [0, 1, 2]:
    ax1.hist(df_wine[df_wine['clase'] == cls]['alcohol'],
             bins=15, alpha=0.55, label=f'Clase {cls}')
ax1.set(title='Distribución: alcohol por clase', xlabel='Alcohol', ylabel='Frecuencia')
ax1.legend()

# Scatter plot flavanoids vs color_intensity
ax2 = fig.add_subplot(gs[0, 1])
scatter = ax2.scatter(df_wine['flavanoids'], df_wine['color_intensity'],
                      c=df_wine['clase'], cmap='Set1', alpha=0.7, edgecolors='k', s=40)
ax2.set(title='Flavanoids vs. Color Intensity',
        xlabel='Flavanoids', ylabel='Color Intensity')
plt.colorbar(scatter, ax=ax2, label='Clase')

# Heatmap de correlación
ax3 = fig.add_subplot(gs[1, :])
corr = df_wine.drop('clase', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.3, ax=ax3, cbar_kws={'shrink': 0.7})
ax3.set_title('Matriz de correlación — Wine dataset')

plt.suptitle('EDA — Wine Dataset', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 7.2 Preprocesamiento y pipeline

In [ ]:
# ── Separar features y target ─────────────────────────────────────────────────
X_wine = df_wine.drop('clase', axis=1)
y_wine = df_wine['clase']

X_w_train, X_w_test, y_w_train, y_w_test = train_test_split(
    X_wine, y_wine, test_size=0.25, random_state=42, stratify=y_wine
)

print(f'Train: {X_w_train.shape} | Test: {X_w_test.shape}')

# ── Pipeline con StandardScaler (todas numéricas) ─────────────────────────────
wine_classifiers = {
    'Logística (C=1)':   Pipeline([('sc', StandardScaler()),
                                   ('clf', LogisticRegression(C=1, multi_class='auto',
                                                              max_iter=1000))]),
    'Logística (C=0.01)': Pipeline([('sc', StandardScaler()),
                                    ('clf', LogisticRegression(C=0.01, multi_class='auto',
                                                               max_iter=1000))]),
    'SVM lineal':        Pipeline([('sc', StandardScaler()),
                                   ('clf', SVC(kernel='linear', C=1, probability=True,
                                               random_state=42))]),
    'SVM RBF':           Pipeline([('sc', StandardScaler()),
                                   ('clf', SVC(kernel='rbf', C=10, probability=True,
                                               random_state=42))]),
    'Random Forest':     Pipeline([('sc', StandardScaler()),
                                   ('clf', RandomForestClassifier(n_estimators=100,
                                                                  random_state=42))]),
    'Gradient Boosting': Pipeline([('sc', StandardScaler()),
                                   ('clf', GradientBoostingClassifier(n_estimators=100,
                                                                      random_state=42))]),
}

print('Pipelines definidos ✓')

### 7.3 Selección de modelos con CV estratificada

In [ ]:
# ── Evaluación con CV 10-fold ─────────────────────────────────────────────────
skf_wine = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
wine_results = {}

for name, clf in wine_classifiers.items():
    scores = cross_val_score(clf, X_w_train, y_w_train,
                             cv=skf_wine, scoring='accuracy', n_jobs=-1)
    wine_results[name] = scores
    print(f'{name:<22}  Accuracy = {scores.mean():.4f} ± {scores.std():.4f}')

# ── Seleccionar mejor modelo ──────────────────────────────────────────────────
best_wine_name = max(wine_results, key=lambda k: wine_results[k].mean())
print(f'\nMejor modelo (CV): {best_wine_name}')

### 7.4 Evaluación final en test

In [ ]:
# ── Entrenar mejor modelo en todos los datos de train y evaluar en test ────────
best_wine_clf = wine_classifiers[best_wine_name]
best_wine_clf.fit(X_w_train, y_w_train)
y_w_pred = best_wine_clf.predict(X_w_test)

print(f'Modelo: {best_wine_name}')
print(f'Exactitud en test: {accuracy_score(y_w_test, y_w_pred):.4f}')
print('\nReporte de clasificación:')
print(classification_report(y_w_test, y_w_pred,
                            target_names=['Clase 0', 'Clase 1', 'Clase 2']))

# ── Matriz de confusión ───────────────────────────────────────────────────────
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(y_w_test, y_w_pred)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Clase 0', 'Clase 1', 'Clase 2'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Matriz de Confusión — {best_wine_name}')
plt.tight_layout()
plt.show()

### 7.5 Interpretación del modelo final

In [ ]:
# ── Feature importance del modelo final ──────────────────────────────────────
final_model = best_wine_clf.named_steps['clf']

if hasattr(final_model, 'feature_importances_'):
    importances_wine = pd.Series(
        final_model.feature_importances_,
        index=X_wine.columns
    ).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    importances_wine.plot.bar(ax=ax, color='mediumseagreen', edgecolor='darkgreen')
    ax.set(title=f'Feature Importances — {best_wine_name}',
           ylabel='Importancia (MDI)', xlabel='')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

elif hasattr(final_model, 'coef_'):
    # Regresión logística: coeficientes (una fila por clase)
    coef_df = pd.DataFrame(
        final_model.coef_,
        columns=X_wine.columns,
        index=['Clase 0', 'Clase 1', 'Clase 2']
    )
    fig, ax = plt.subplots(figsize=(12, 4))
    coef_df.T.plot.bar(ax=ax, edgecolor='black', alpha=0.8)
    ax.set(title='Coeficientes — Regresión Logística',
           ylabel='Coeficiente', xlabel='')
    ax.tick_params(axis='x', rotation=45)
    ax.axhline(0, color='black', lw=0.8, ls='--')
    plt.tight_layout()
    plt.show()

# ── Permutation importance sobre test ─────────────────────────────────────────
perm_wine = permutation_importance(
    best_wine_clf, X_w_test, y_w_test,
    n_repeats=15, random_state=42, scoring='accuracy'
)
perm_wine_df = pd.DataFrame({
    'feature':    X_wine.columns,
    'importance': perm_wine.importances_mean,
    'std':        perm_wine.importances_std
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(perm_wine_df['feature'][::-1],
        perm_wine_df['importance'][::-1],
        xerr=perm_wine_df['std'][::-1],
        color='mediumpurple', edgecolor='indigo', capsize=4)
ax.axvline(0, color='black', lw=0.8, ls='--')
ax.set(title=f'Permutation Importance (test) — {best_wine_name}',
       xlabel='Reducción en Accuracy')
plt.tight_layout()
plt.show()

### 7.6 Visualización 2D con reducción de dimensionalidad (PCA)

In [ ]:
from sklearn.decomposition import PCA

# ── PCA para visualización ────────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
scaler_vis = StandardScaler()
X_wine_scaled = scaler_vis.fit_transform(X_wine)
X_wine_pca    = pca.fit_transform(X_wine_scaled)

var_exp = pca.explained_variance_ratio_

# ── Fronteras de decisión en el espacio PCA ───────────────────────────────────
# Entrenamos un clasificador simple en el espacio PCA para visualización
rf_pca = RandomForestClassifier(n_estimators=100, random_state=42)
rf_pca.fit(X_wine_pca, y_wine)

x_min, x_max = X_wine_pca[:, 0].min() - 0.5, X_wine_pca[:, 0].max() + 0.5
y_min, y_max = X_wine_pca[:, 1].min() - 0.5, X_wine_pca[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
Z = rf_pca.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter PCA
colors_cls = ['#e74c3c', '#2ecc71', '#3498db']
for cls, color in zip([0, 1, 2], colors_cls):
    mask_cls = y_wine == cls
    axes[0].scatter(X_wine_pca[mask_cls, 0], X_wine_pca[mask_cls, 1],
                    c=color, label=f'Clase {cls}', alpha=0.7, edgecolors='k', s=50)
axes[0].set(title=f'PCA — Wine dataset\n(Var. explicada: PC1={var_exp[0]:.1%}, PC2={var_exp[1]:.1%})',
            xlabel='PC1', ylabel='PC2')
axes[0].legend()

# Fronteras de decisión
axes[1].contourf(xx, yy, Z, alpha=0.25, cmap=plt.cm.RdYlBu)
axes[1].contour(xx, yy, Z, colors='k', linewidths=0.4, alpha=0.4)
for cls, color in zip([0, 1, 2], colors_cls):
    mask_cls = y_wine == cls
    axes[1].scatter(X_wine_pca[mask_cls, 0], X_wine_pca[mask_cls, 1],
                    c=color, label=f'Clase {cls}', alpha=0.8, edgecolors='k', s=50)
axes[1].set(title='Fronteras de decisión (RF en espacio PCA)',
            xlabel='PC1', ylabel='PC2')
axes[1].legend()

plt.suptitle('Visualización 2D — Wine Dataset', fontsize=13)
plt.tight_layout()
plt.show()

### 7.7 Resumen del mini-proyecto

In [ ]:
# ── Tabla resumen del mini-proyecto ──────────────────────────────────────────
summary = pd.DataFrame({
    'Etapa': [
        'Dataset', 'EDA', 'Preprocesamiento',
        'Modelos evaluados', 'Mejor modelo (CV)',
        'Accuracy en test', 'Variables más importantes'
    ],
    'Detalle': [
        'Wine (178 muestras, 13 features, 3 clases)',
        'Histogramas, scatter, correlación',
        'StandardScaler dentro de pipeline',
        str(len(wine_classifiers)),
        best_wine_name,
        f'{accuracy_score(y_w_test, y_w_pred):.4f}',
        ', '.join(perm_wine_df.head(3)['feature'].tolist())
    ]
})
display(summary.set_index('Etapa'))

# ── Guardar modelo final ──────────────────────────────────────────────────────
wine_model_path = os.path.join(tempfile.gettempdir(), 'modelo_final_wine.pkl')
joblib.dump(best_wine_clf, wine_model_path)
print(f'\nModelo final serializado: {wine_model_path}')

---

## 8. Consolidación: Mapa del Curso

```
MINERÍA DE DATOS — Ciclo completo
═══════════════════════════════════════════════════════════════════════

  Datos crudos
      │
      ▼
  [S1]  Preprocesamiento ─── imputación, escalado, encoding
      │
      ▼
  [S7]  Feature Engineering ─── selección, construcción, PCA/LDA
      │
      ▼
  [S3]  Partición y evaluación ─── CV, métricas, curvas ROC
      │
      ├──▶ [S2]  Modelos lineales (Regresión Logística, Ridge, Lasso)
      │
      ├──▶ [S4]  Regularización (Elastic Net, control de complejidad)
      │
      ├──▶ [S5]  SVM, Árboles de Decisión
      │
      ├──▶ [S6]  Ensambles (Bagging, Boosting, Stacking)
      │
      ├──▶ [S8–9] Clustering (K-Means, DBSCAN, Jerárquico)
      │
      ├──▶ [S10] Detección de anomalías (Isolation Forest, LOF)
      │
      └──▶ [S11] Reducción de dimensionalidad (PCA, t-SNE, UMAP)
      │
      ▼
  [S12] Interpretabilidad + Pipelines + Buenas prácticas
      │
      ▼
  Modelo listo para producción
═══════════════════════════════════════════════════════════════════════
```

---

## 9. Ejercicios Propuestos (Pre-Parcial 3)

Los siguientes ejercicios cubren los temas del Parcial 3. Se recomienda resolverlos **sin mirar el cuaderno** primero.

---

### Ejercicio 1 — Pipeline completo

Cargue el dataset `load_wine()`. Construya un `Pipeline` que incluya:
1. `SimpleImputer` con estrategia `median`.
2. `StandardScaler`.
3. `SVC` con kernel RBF.

Evalúelo con CV de 10 folds (métrica: `accuracy`) y reporte media y desviación estándar.

---

### Ejercicio 2 — Selección estadística de modelos

Compare un `LogisticRegression` y un `GradientBoostingClassifier` en el dataset `load_breast_cancer()` usando CV de 10 folds. Aplique un t-test pareado y concluya si la diferencia es estadísticamente significativa (α = 0.05).

---

### Ejercicio 3 — Interpretabilidad

Usando un `RandomForestClassifier` entrenado en `load_wine()`:
1. Calcule la importancia MDI y grafíquela.
2. Calcule la Permutation Importance en el conjunto de test.
3. Compare ambas listas. ¿Cambia el ranking de las top-3 variables? ¿A qué puede deberse?

---

### Ejercicio 4 — Data Leakage

En el siguiente código, identifique y corrija el error de data leakage:

```python
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # ← ¿dónde está el problema?

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3)

clf = LogisticRegression()
clf.fit(X_train, y_train)
print(clf.score(X_test, y_test))
```

---

### Ejercicio 5 — Mini-proyecto integrador

Use `fetch_california_housing()` (regresión):
1. Realice un EDA (distribuciones, correlaciones, valores atípicos).
2. Construya un pipeline con preprocesamiento completo.
3. Compare al menos 3 modelos de regresión con CV.
4. Evalúe el mejor modelo en test (RMSE, R²).
5. Grafique la Permutation Importance.
6. Grafique el PDP de las 2 variables más importantes.

---

## 10. Espacio de trabajo — Ejercicios

Utilice las siguientes celdas para resolver los ejercicios propuestos.

In [ ]:
# ── Ejercicio 1 — Pipeline completo ──────────────────────────────────────────
# Escriba su solución aquí


In [ ]:
# ── Ejercicio 2 — t-test pareado ─────────────────────────────────────────────
# Escriba su solución aquí


In [ ]:
# ── Ejercicio 3 — Interpretabilidad ──────────────────────────────────────────
# Escriba su solución aquí


In [ ]:
# ── Ejercicio 4 — Corrección de data leakage ─────────────────────────────────
# Escriba su solución aquí


In [ ]:
# ── Ejercicio 5 — Mini-proyecto integrador ────────────────────────────────────
# Escriba su solución aquí


---

## Conclusiones

Este cuaderno ha recorrido los tópicos avanzados y prácticos que un científico de datos encontrará en proyectos reales:

1. **Pipelines**: la forma profesional de encapsular el flujo completo, evitar leakage y facilitar el despliegue.

2. **Comparación rigurosa de modelos**: los números solos no bastan; el t-test pareado nos dice si las diferencias son reales o producto del azar muestral.

3. **Interpretabilidad**: un modelo explicable genera confianza, facilita la auditoría y permite corregir errores sistémicos. Usar MDI, Permutation Importance y SHAP según el contexto.

4. **Data drift**: los modelos en producción se degradan. Monitorear la distribución de las features con tests estadísticos (KS, Mann-Whitney) es esencial.

5. **Errores comunes**: el data leakage es silencioso y devastador. Un buen pipeline lo previene estructuralmente.

---

### Referencias recomendadas

- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3ª ed.). O'Reilly.
- Lundberg, S. & Lee, S.-I. (2017). A Unified Approach to Interpreting Model Predictions. *NeurIPS*.
- Dietterich, T. (1998). Approximate Statistical Tests for Comparing Supervised Classification Learning Algorithms. *Neural Computation*.
- Scikit-learn documentation: https://scikit-learn.org/stable/
- SHAP documentation: https://shap.readthedocs.io/

---

*Universidad de los Andes — Minería de Datos — Semana 12*  
*Cuaderno preparado para la Sesión 23 (Tópicos Avanzados) y la preparación del Parcial 3*